# LangChain Tools

In [6]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
# from langchain_openai import ChatOpenAI
load_dotenv()

model = ChatGroq(model="openai/gpt-oss-120b")
model.invoke("What is my name?")

AIMessage(content='I don’t have any information about your name. If you’d like me to address you by a particular name, just let me know!', additional_kwargs={'reasoning_content': 'The user asks "What is my name?" There\'s no prior context. According to policy, we must not guess personal info. We should respond that we don\'t know the name. Ask if they\'d like to share.'}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 76, 'total_tokens': 156, 'completion_time': 0.168881837, 'completion_tokens_details': {'reasoning_tokens': 43}, 'prompt_time': 0.026119589, 'prompt_tokens_details': None, 'queue_time': 0.38448574, 'total_time': 0.195001426}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_96d96a151c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a20c5e-e773-70c7-a672-8e7de2e4a088-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'output_tokens': 80, 

In [7]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}"


In [9]:
model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke(
    "What's the weather like in Boston?"
)

print(response)

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


content='' additional_kwargs={'reasoning_content': 'The user asks current weather in Boston. We need to call get_weather function with location "Boston".', 'tool_calls': [{'id': 'fc_6ed1b17f-04d9-4074-8798-2d814f4af87e', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 127, 'total_tokens': 175, 'completion_time': 0.100035871, 'completion_tokens_details': {'reasoning_tokens': 21}, 'prompt_time': 0.006799818, 'prompt_tokens_details': None, 'queue_time': 0.309540804, 'total_time': 0.106835689}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f640395b96', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a20c60-7dac-769e-877e-15f797b127ff-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_6ed1b17f-04d9-4074-8798-2d814f4af87e', 'type': 'tool_call'}] invalid

In [11]:
# Step 1: Model generates tool calls
messages = [
    {"role": "user", "content": "What's the weather in Boston?"}
]

ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)
print(messages)

# Step 2: Execute tools
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

print(messages)
# Step 3: Get the final response
final_response = model_with_tools.invoke(messages)

print(final_response.text)


[{'role': 'user', 'content': "What's the weather in Boston?"}, AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to fetch weather via function. Use get_weather with location "Boston".', 'tool_calls': [{'id': 'fc_887bdd2e-2435-48fb-80bd-8ebd91e334f8', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 126, 'total_tokens': 170, 'completion_time': 0.093615811, 'completion_tokens_details': {'reasoning_tokens': 17}, 'prompt_time': 0.0047365, 'prompt_tokens_details': None, 'queue_time': 0.247213199, 'total_time': 0.098352311}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_47082602e2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a20c64-7945-7760-95bd-8fb46e771340-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_887bdd2e-2435-48